# Day 7 — Ring evaluation

**This is the first notebook that uses `isFraud` for anything graph-related** — and it's used only to *evaluate* the clusters built blind on Days 5-6, never to build them. See the discussion in [docs/identity-graph.md](../docs/identity-graph.md) for why that separation is what makes this evaluation genuine rather than circular.

In [1]:
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_merged_train
from src.graph import load_graph
from src.ring_eval import cluster_stats, evaluate_flagged_clusters, classify_cluster_edge_types

train = load_merged_train()

identity_graph = load_graph('../data/processed/identity_graph.pkl')
behavioral_graph = load_graph('../data/processed/behavioral_graph.pkl')

with open('../data/processed/cluster_partition.json') as f:
    partition_raw = json.load(f)
partition = {int(k): v for k, v in partition_raw.items()}

print('Loaded', len(partition), 'transactions across', len(set(partition.values())), 'clusters')

Loaded 590540 transactions across 265011 clusters


In [2]:
train['cluster_id'] = train['TransactionID'].map(partition)

population_fraud_rate = train['isFraud'].mean()
total_fraud = int(train['isFraud'].sum())
print(f'Population fraud rate: {population_fraud_rate:.4f} ({total_fraud:,} fraud transactions total)')

stats = cluster_stats(train['TransactionID'], train['cluster_id'], train['isFraud'])
non_trivial_stats = stats[stats['size'] > 1]
print(f'Non-trivial clusters: {len(non_trivial_stats):,}')

Population fraud rate: 0.0350 (20,663 fraud transactions total)


Non-trivial clusters: 40,451


## 1. Did blind clustering actually find fraud?

"Flagged" = every transaction inside a non-trivial cluster is treated as a positive prediction. This is the headline check: does grouping by shared card/address/email/behavior — done with zero knowledge of which transactions are fraud — actually concentrate real fraud?

In [3]:
result_min2 = evaluate_flagged_clusters(stats, min_size=2, total_fraud=total_fraud, population_fraud_rate=population_fraud_rate)

print(f"Clusters flagged (size >= 2): {result_min2['n_clusters_flagged']:,}")
print(f"Transactions flagged: {result_min2['n_transactions_flagged']:,}")
print(f"Fraud captured: {result_min2['n_fraud_captured']:,}")
print(f"Precision (fraud % within flagged clusters): {result_min2['precision']:.4f}")
print(f"Fraud lift vs. population baseline ({population_fraud_rate:.4f}): {result_min2['lift']:.2f}x")
print(f"Capture rate (% of ALL fraud found inside flagged clusters): {result_min2['capture_rate']:.4f}")

Clusters flagged (size >= 2): 40,451
Transactions flagged: 365,980
Fraud captured: 15,151
Precision (fraud % within flagged clusters): 0.0414
Fraud lift vs. population baseline (0.0350): 1.18x
Capture rate (% of ALL fraud found inside flagged clusters): 0.7332


## 2. Sensitivity check: does this hold at different ring-size thresholds?

A finding that only shows up at exactly one arbitrary cutoff isn't trustworthy. Cluster size is known before any label is looked at, so it's used here as the ring-definition threshold — checking min sizes 2, 3, and 5.

In [4]:
sensitivity = pd.DataFrame([
    evaluate_flagged_clusters(stats, min_size=m, total_fraud=total_fraud, population_fraud_rate=population_fraud_rate)
    for m in [2, 3, 5]
]).set_index('min_size')

sensitivity

,n_clusters_flagged,n_transactions_flagged,n_fraud_captured,precision,lift,capture_rate
min_size,,,,,,
2,40451,365980,15151,0.041398,1.183150,0.733243
3,17499,320076,13374,0.041784,1.194164,0.647244
5,6629,284079,11960,0.042101,1.203228,0.578812


## 3. Does the behavioral layer earn its place?

Day 6 showed 94.2% of behavioral edges connect transactions with no shared identity attribute — a structural finding. Now check the part that actually matters: do clusters found **only** through behavioral similarity (no identity edge inside them at all) still concentrate real fraud, or is that similarity coincidental?

In [5]:
edge_types = classify_cluster_edge_types(partition, identity_graph, behavioral_graph)

non_trivial_stats = non_trivial_stats.copy()
non_trivial_stats['has_identity'] = non_trivial_stats.index.map(lambda c: edge_types['has_identity'].get(c, False))
non_trivial_stats['has_behavioral'] = non_trivial_stats.index.map(lambda c: edge_types['has_behavioral'].get(c, False))

def category(row):
    if row['has_identity'] and row['has_behavioral']:
        return 'both'
    if row['has_identity']:
        return 'identity_only'
    if row['has_behavioral']:
        return 'behavioral_only'
    return 'neither'

non_trivial_stats['category'] = non_trivial_stats.apply(category, axis=1)

summary = non_trivial_stats.groupby('category').apply(
    lambda g: pd.Series({
        'n_clusters': len(g),
        'n_transactions': g['size'].sum(),
        'n_fraud': g['fraud_count'].sum(),
        'fraud_rate': g['fraud_count'].sum() / g['size'].sum(),
    }),
    include_groups=False,
)
summary['lift_vs_population'] = summary['fraud_rate'] / population_fraud_rate
summary

,n_clusters,n_transactions,n_fraud,fraud_rate,lift_vs_population
category,,,,,
behavioral_only,18485.0,45662.0,2402.0,0.052604,1.503398
both,8854.0,285483.0,12053.0,0.042220,1.206621
identity_only,13112.0,34835.0,696.0,0.019980,0.571017


In [6]:
behavioral_only_count = (non_trivial_stats['category'] == 'behavioral_only').sum()
print(f"Clusters found ONLY via behavioral similarity (no shared identifier at all): {behavioral_only_count:,}")
print(f"Their combined fraud rate: {summary.loc['behavioral_only', 'fraud_rate']:.4f} "
      f"({summary.loc['behavioral_only', 'lift_vs_population']:.2f}x population baseline)" 
      if 'behavioral_only' in summary.index else 'No behavioral-only clusters found.')

Clusters found ONLY via behavioral similarity (no shared identifier at all): 18,485
Their combined fraud rate: 0.0526 (1.50x population baseline)


## 4. Manual false-positive audit

A handful of larger flagged clusters with **zero** fraud, inspected by hand for a plausible innocent explanation rather than assumed to be a system failure.

In [7]:
false_positive_clusters = non_trivial_stats[non_trivial_stats['fraud_rate'] == 0].sort_values('size', ascending=False)
print(f'Non-trivial clusters with zero fraud: {len(false_positive_clusters):,} of {len(non_trivial_stats):,}')

audit_cols = ['TransactionID', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'addr1', 'P_emaildomain', 'TransactionDT']

for cluster_id in false_positive_clusters.head(5).index:
    members = train[train['cluster_id'] == cluster_id][audit_cols].sort_values('TransactionDT')
    print(f'\n--- Cluster {cluster_id} (size {len(members)}, fraud_rate 0) ---')
    print(members.to_string(index=False))

Non-trivial clusters with zero fraud: 36,866 of 40,451

--- Cluster 13673 (size 411, fraud_rate 0) ---
 TransactionID  TransactionAmt ProductCD  card1  card2  addr1 P_emaildomain  TransactionDT
       2994974          26.000         W   7638  174.0  330.0     gmail.com         246348
       2995575          93.950         W   7638  174.0  330.0     yahoo.com         254998
       2996933         300.000         W   7638  174.0  123.0   hotmail.com         310969
       2997188          78.500         W  10112  360.0  299.0     gmail.com         316827
       2997206          87.000         W   2455  321.0  441.0     gmail.com         317154
       2997208         100.000         H  13844  583.0  315.0    icloud.com         317195
       2997210         120.000         H  16659  170.0  325.0       aol.com         317231
       2997225          66.950         W   4806  490.0  315.0       aol.com         317466
       2997461          50.000         S   6741  583.0  330.0           NaN   


--- Cluster 38430 (size 264, fraud_rate 0) ---
 TransactionID  TransactionAmt ProductCD  card1  card2  addr1  P_emaildomain  TransactionDT
       3075635          75.000         R  11516  320.0  269.0    hotmail.com        1872529
       3078892          59.000         W  11516  320.0  269.0      gmail.com        1902154
       3079495          29.000         W  18148  516.0  299.0      gmail.com        1909258
       3080397         201.000         W  17466  225.0  264.0      gmail.com        1941824
       3082030         200.000         R  11516  320.0  191.0      yahoo.com        1961076
       3082550         201.000         W  17466  225.0  264.0      gmail.com        1964391
       3084430         335.000         W  12906  516.0  264.0      gmail.com        1977713
       3084619         107.950         W   9197  555.0  498.0      yahoo.com        1979143
       3084698          57.950         W   9197  555.0  498.0      yahoo.com        1979833
       3084714         107.950  

 TransactionID  TransactionAmt ProductCD  card1  card2  addr1  P_emaildomain  TransactionDT
       3080057         107.950         W   5700  122.0  264.0      gmail.com        1918778
       3083234          57.950         W   5700  122.0  272.0      gmail.com        1968964
       3085100          59.000         W   1047  560.0  492.0    hotmail.com        1983123
       3085741         170.950         W   7720  514.0  299.0      yahoo.com        1988331
       3085905         226.000         W   5700  122.0  512.0      gmail.com        1989837
       3086801          57.950         W   5700  122.0  204.0      gmail.com        2002060
       3086976          30.782         C  15885  545.0    NaN    outlook.com        2005443
       3086977          39.000         W   5700  122.0  191.0      gmail.com        2005494
       3086979          34.630         C   9633  296.0    NaN      gmail.com        2005560
       3087302         335.000         W  14059  492.0  410.0      yahoo.com    

## Save ring evaluation results

In [8]:
ring_results = {
    'population_fraud_rate': population_fraud_rate,
    'total_fraud': total_fraud,
    'headline_min_size_2': result_min2,
    'sensitivity_by_min_size': sensitivity.reset_index().to_dict(orient='records'),
    'edge_type_breakdown': summary.reset_index().to_dict(orient='records'),
    'behavioral_only_cluster_count': int(behavioral_only_count),
    'false_positive_cluster_count': int(len(false_positive_clusters)),
}

with open('../results/ring_evaluation_metrics.json', 'w') as f:
    json.dump(ring_results, f, indent=2, default=str)

print('Saved to results/ring_evaluation_metrics.json')

Saved to results/ring_evaluation_metrics.json


## Takeaway

_Fill in after running: the fraud lift and capture rate at the headline threshold, whether the finding holds across all three sensitivity thresholds, whether behavioral-only clusters actually concentrate fraud (the number that justifies signal 3), and one honest note on what the false-positive audit revealed._